# 08b — Batch Evaluation: All Full-Game Videos × 4 Tracker Configs

Runs every combination of detector+tracker against every full-game video in
`data/processed/labelstudio/`, computes GT-vs-tracker posterior divergence
and icon count accuracy, and reports aggregate metrics in summary tables.

**Outputs per run configuration:**
- Mean posterior divergence across all videos
- Mean per-video max divergence
- Mean final-frame divergence
- Aggregated icon count table (summed over all videos)

No per-video plots are produced — this is a batch metrics notebook.

Prerequisites: `02_train_yolo.ipynb` · `03_train_rtdetr.ipynb` · `04_tracker_architecture_botsort.ipynb` · `04b_tracker_architecture_bytetrack.ipynb`

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, cv2, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from ultralytics import YOLO

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')
RUNS_DIR     = (REPO_ROOT / 'runs').resolve()
MOT_SEQ_DIR  = REPO_ROOT / 'data/mot_dataset/sequences'
CHUNKS_DIR   = REPO_ROOT / 'data/processed/labelstudiochunks'
OUT_DIR      = (REPO_ROOT / 'runs/longform').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Select the full game to analyse ──────────────────────────────────────
# Set to the shared prefix of all chunks belonging to one game.
# All sequences whose name starts with this prefix will be used in order.
GAME_PREFIX = 'AY_G1'    # e.g. 'AY_G1', 'SerpentBoi_G2', 'Frozoha_G1', ...

# ── Detector weights ──────────────────────────────────────────────────────
WEIGHTS     = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
# WEIGHTS   = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'

# ── Tracker config: baseline arch config from notebook 04 ────────────────
# HP tuning (notebooks 05/05b) was abandoned — baseline outperforms tuned configs.
_ARCH_CFG_PATH = REPO_ROOT / 'configs' / 'kartector_botsort_arch.json'
if _ARCH_CFG_PATH.exists():
    import json as _jcfg08
    _arch_cfg08   = _jcfg08.loads(_ARCH_CFG_PATH.read_text())
    TRACKER_CFG   = str(REPO_ROOT / _arch_cfg08.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml'))
    CONF          = _arch_cfg08.get('conf', 0.25)
    IOU           = 0.70
    REID_OPTION   = _arch_cfg08.get('reid_option', 'B')
    REID_B_PARAMS = _arch_cfg08.get('reid_b_params', {})
else:
    TRACKER_CFG   = str(REPO_ROOT / 'configs' / 'kartector_botsort_reentry.yaml')
    CONF = 0.25; IOU = 0.70
    REID_OPTION   = 'B'
    REID_B_PARAMS = {}
if not Path(TRACKER_CFG).exists():
    TRACKER_CFG = 'botsort.yaml'

def _make_boxmot_tracker08():
    """Instantiate a fresh BotSort tracker using the baseline arch config."""
    import torch
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    device = torch.device(
        'cuda' if torch.cuda.is_available() else
        'mps'  if torch.backends.mps.is_available() else 'cpu')
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=device, half=False,
        track_buffer=REID_B_PARAMS.get('track_buffer', 30),
        match_thresh=REID_B_PARAMS.get('match_thresh', 0.70),
        appearance_thresh=REID_B_PARAMS.get('appearance_thresh', 0.40),
        proximity_thresh=REID_B_PARAMS.get('proximity_thresh', 0.5),
        with_reid=True,
    )

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES   = splits['classes']
N_CLASSES = len(CLASSES)

# Discover all sequences for this game, in chunk order
all_seqs = sorted(MOT_SEQ_DIR.iterdir())
game_seqs = [s for s in all_seqs if s.name.startswith(GAME_PREFIX) and s.is_dir()]
print(f'Game prefix : {GAME_PREFIX}')
print(f'Chunks found: {len(game_seqs)}')
print(f'Weights     : {WEIGHTS}')
print(f'Tracker     : {TRACKER_CFG}')

## 1 — Minigame Model

`compute_posterior` is imported from `helpers.py`.

In [ ]:
from helpers import compute_posterior

MINIGAMES = [
    "Single Race",
    "Drag Race",
    "Destruction Derby",
]
PRIOR = np.ones(len(MINIGAMES)) / len(MINIGAMES)
LIKELIHOODS = np.array([
    [0.083,0.083,0.083,0.083,0.075,0.062,0.083,0.083,0.083],
    [0.082,0.082,0.082,0.082,0.074,0.082,0.082,0.082,0.074],
    [0.09, 0.072,0.108,0.072,0.072,0.129,0.05, 0.072,0.072],
])
LIKELIHOODS = LIKELIHOODS / LIKELIHOODS.sum(axis=1, keepdims=True)
assert LIKELIHOODS.shape == (len(MINIGAMES), N_CLASSES)

## 2 — Helpers

`load_gt_for_video` and `accumulate_longform` are imported from `helpers.py`,
alongside `merge_fragments` used in the tracker infrastructure.

In [ ]:
from helpers import merge_fragments, accumulate_longform

## 3 — Tracker Infrastructure

In [ ]:
import time, numpy as _np8c
from ultralytics import YOLO as _YOLO8c

def _make_boxmot_8c(reid_b_params):
    import torch
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    device = torch.device('cuda' if torch.cuda.is_available() else
                          'mps'  if torch.backends.mps.is_available() else 'cpu')
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=device, half=False,
        track_buffer=reid_b_params.get('track_buffer', 30),
        match_thresh=reid_b_params.get('match_thresh', 0.70),
        appearance_thresh=reid_b_params.get('appearance_thresh', 0.40),
        proximity_thresh=reid_b_params.get('proximity_thresh', 0.5),
        with_reid=True,
    )


def run_full_video(run_cfg, video_path, conf, iou, n_classes):
    """Run one model×tracker on a full video. Returns (df, elapsed_s)."""
    det  = _YOLO8c(str(run_cfg['weights']))
    rows = []
    t0   = time.time()
    if run_cfg['tracker'] == 'botsort' and run_cfg['reid_option'] == 'B':
        _trk = _make_boxmot_8c(run_cfg['reid_b_params'])
        _cap = cv2.VideoCapture(str(video_path))
        _fi  = 0
        while True:
            _ret, _frame = _cap.read()
            if not _ret: break
            _res  = det(_frame, conf=conf, verbose=False)
            _dets = _res[0].boxes
            _darr = _np8c.concatenate([
                _dets.xyxy.cpu().numpy(),
                _dets.conf.cpu().numpy().reshape(-1,1),
                _dets.cls.cpu().numpy().reshape(-1,1)], axis=1) \
                if (_dets is not None and len(_dets)) else _np8c.empty((0,6))
            for _t in _trk.update(_darr, _frame):
                _cid = int(_t[6]) if len(_t) > 6 else 0
                if 0 <= _cid < n_classes:
                    rows.append({'frame': _fi, 'track_id': int(_t[4]),
                                 'class_id': _cid, 'conf': float(_t[5])})
            _fi += 1
        _cap.release()
    else:
        for _fi, r in enumerate(
                det.track(str(video_path), tracker=run_cfg['tracker_cfg'],
                          conf=conf, iou=iou, stream=True,
                          verbose=False, persist=False)):
            if r.boxes is not None and r.boxes.id is not None:
                for tid, cls, c in zip(
                        r.boxes.id.cpu().numpy().astype(int),
                        r.boxes.cls.cpu().numpy().astype(int),
                        r.boxes.conf.cpu().numpy()):
                    if 0 <= int(cls) < n_classes:
                        rows.append({'frame': _fi, 'track_id': int(tid),
                                     'class_id': int(cls), 'conf': float(c)})
    elapsed = time.time() - t0
    _e = pd.DataFrame(columns=['frame','track_id','class_id','conf'])
    return (pd.DataFrame(rows) if rows else _e), elapsed


# ── Define 4 run configurations ───────────────────────────────────────────
RTDETR_WEIGHTS = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'
BYTETRACK_CFG  = str(REPO_ROOT / 'configs' / 'kartector_bytetrack.yaml')
if not Path(BYTETRACK_CFG).exists(): BYTETRACK_CFG = 'bytetrack.yaml'

RUNS_TO_EVAL = [
    {'label': 'YOLO + BoTSORT',      'weights': WEIGHTS,        'tracker': 'botsort',   'tracker_cfg': TRACKER_CFG,   'reid_option': REID_OPTION, 'reid_b_params': REID_B_PARAMS, 'available': WEIGHTS.exists()},
    {'label': 'YOLO + ByteTrack',    'weights': WEIGHTS,        'tracker': 'bytetrack', 'tracker_cfg': BYTETRACK_CFG, 'reid_option': 'none',      'reid_b_params': {},            'available': WEIGHTS.exists()},
    {'label': 'RT-DETR + BoTSORT',   'weights': RTDETR_WEIGHTS, 'tracker': 'botsort',   'tracker_cfg': TRACKER_CFG,   'reid_option': REID_OPTION, 'reid_b_params': REID_B_PARAMS, 'available': RTDETR_WEIGHTS.exists()},
    {'label': 'RT-DETR + ByteTrack', 'weights': RTDETR_WEIGHTS, 'tracker': 'bytetrack', 'tracker_cfg': BYTETRACK_CFG, 'reid_option': 'none',      'reid_b_params': {},            'available': RTDETR_WEIGHTS.exists()},
]
for _r in RUNS_TO_EVAL:
    print(f"  {'✓' if _r['available'] else '✗'} {_r['label']}")

# ── Discover all full-game videos ────────────────────────────────────────
FULL_VIDEO_DIR = REPO_ROOT / 'data' / 'processed' / 'labelstudio'
all_videos = sorted(FULL_VIDEO_DIR.glob('*.mp4'))
print(f'\nFull-game videos: {len(all_videos)}')
for v in all_videos: print(f'  {v.name}')


## 4 — Batch Run: All Videos × All Configs

For each video, loads GT, runs all 4 tracker configs, accumulates posteriors,
and records per-video divergence metrics and icon counts.
No plots are produced here — see Section 5 for the summary tables.

In [ ]:
# all_results[run_label][video_stem] = {
#   'elapsed', 'mean_div', 'max_div', 'final_div',
#   'trk_counts', 'gt_counts'
# }
all_results   = {r['label']: {} for r in RUNS_TO_EVAL if r['available']}
skipped_videos = []

for video_path in tqdm(all_videos, desc='Videos'):
    game_prefix = video_path.stem   # e.g. 'AY_G1'

    # ── Load GT ────────────────────────────────────────────────────────────
    gt_df, total_frames, _ = load_gt_for_video(game_prefix, MOT_SEQ_DIR, N_CLASSES)
    if gt_df is None or total_frames == 0:
        print(f'  [SKIP] no GT sequences found for {game_prefix}')
        skipped_videos.append(game_prefix)
        continue
    gt_posts, gt_counts = accumulate_longform(
        gt_df, N_CLASSES, total_frames, PRIOR, LIKELIHOODS)

    # ── Run each tracker config ────────────────────────────────────────────
    for run_cfg in RUNS_TO_EVAL:
        lbl = run_cfg['label']
        if not run_cfg['available']: continue

        trk_df, elapsed = run_full_video(run_cfg, video_path, CONF, IOU, N_CLASSES)
        trk_posts, trk_counts = accumulate_longform(
            trk_df, N_CLASSES, total_frames, PRIOR, LIKELIHOODS)

        div = _np8c.abs(gt_posts - trk_posts).sum(axis=1) / 2   # (total_frames,)

        all_results[lbl][game_prefix] = {
            'elapsed':    elapsed,
            'mean_div':   float(div.mean()),
            'max_div':    float(div.max()),
            'final_div':  float(div[-1]),
            'trk_counts': trk_counts.copy(),
            'gt_counts':  gt_counts.copy(),
        }

    print(f'  {game_prefix}: done ({total_frames} frames, {len(gt_df)} GT dets)')

print(f'\nFinished. Skipped: {skipped_videos or "none"}')


## 5 — Summary Tables

### 5a — Divergence Metrics
One row per run config; metrics averaged over all evaluated videos.

### 5b — Icon Count Tables
Four tables (one per run config) with icon counts summed across all videos.

In [ ]:
# ── 5a: Divergence summary table ─────────────────────────────────────────
summary_rows = []
for run_cfg in RUNS_TO_EVAL:
    lbl = run_cfg['label']
    if lbl not in all_results or not all_results[lbl]: continue
    vids = all_results[lbl]
    summary_rows.append({
        'Run':                lbl,
        'Videos evaluated':   len(vids),
        'Mean divergence':    round(_np8c.mean([v['mean_div']  for v in vids.values()]), 4),
        'Mean max divergence':round(_np8c.mean([v['max_div']   for v in vids.values()]), 4),
        'Mean final divergence': round(_np8c.mean([v['final_div'] for v in vids.values()]), 4),
        'Mean elapsed (s)':   round(_np8c.mean([v['elapsed']   for v in vids.values()]), 1),
    })

df_summary = pd.DataFrame(summary_rows).set_index('Run')
print('=== Divergence Summary ===')
display(df_summary.style
    .format({'Mean divergence': '{:.4f}', 'Mean max divergence': '{:.4f}',
             'Mean final divergence': '{:.4f}', 'Mean elapsed (s)': '{:.1f}'})
    .background_gradient(subset=['Mean divergence'],       cmap='YlOrRd')
    .background_gradient(subset=['Mean max divergence'],   cmap='YlOrRd')
    .background_gradient(subset=['Mean final divergence'], cmap='YlOrRd')
    .background_gradient(subset=['Mean elapsed (s)'],      cmap='Blues'))

OUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(OUT_DIR / 'batch_eval_divergence_summary.csv')
print(f'Saved: {OUT_DIR}/batch_eval_divergence_summary.csv')

# ── 5b: Icon count tables (one per run config) ────────────────────────────
for run_cfg in RUNS_TO_EVAL:
    lbl = run_cfg['label']
    if lbl not in all_results or not all_results[lbl]: continue
    vids = all_results[lbl]

    total_gt  = _np8c.zeros(N_CLASSES, dtype=int)
    total_trk = _np8c.zeros(N_CLASSES, dtype=int)
    for v in vids.values():
        total_gt  += v['gt_counts'].astype(int)
        total_trk += v['trk_counts'].astype(int)

    df_counts = pd.DataFrame({
        'GT (total)':      total_gt,
        'Tracker (total)': total_trk,
        'Difference':      total_trk - total_gt,
        '% Error':         _np8c.where(total_gt > 0,
                               (total_trk - total_gt) / total_gt * 100,
                               float('nan')),
    }, index=CLASSES)
    df_counts.loc['TOTAL'] = [
        int(total_gt.sum()), int(total_trk.sum()),
        int((total_trk - total_gt).sum()),
        float((total_trk.sum() - total_gt.sum()) / total_gt.sum() * 100)
        if total_gt.sum() > 0 else float('nan')]

    print(f'\n=== Icon Counts — {lbl} (summed over {len(vids)} videos) ===')
    display(df_counts.style
        .format({'% Error': '{:+.1f}%'})
        .background_gradient(subset=['Difference'], cmap='RdYlGn_r'))
    _slug = lbl.replace(' ', '_').replace('+', 'x')
    df_counts.to_csv(OUT_DIR / f'batch_eval_counts_{_slug}.csv')
    print(f'  Saved: {OUT_DIR}/batch_eval_counts_{_slug}.csv')
